# Primus F1 — download 2025 and 2026 locally

Run this notebook in **primusEnv** from the project checkout. It collects all selected
completed race-weekend sessions: practice, qualifying, sprint qualifying/shootout,
sprint and Grand Prix. Future sessions are excluded. Change `YEARS` later to expand.

The download functions and batch loop are visible below. They reuse this project's
tested SQLite/Parquet writer, FastF1 normalizer and HTTP pacing library; they do not
invoke the old command-line collector or start a hidden process.

**Workflow:** setup → local F1 TV login → preview calendar → one-session pilot →
inspect coverage → full batch → validate → Streamlit phase 1.
Set `RUN_DOWNLOAD = True` only when ready. Start with `SESSION_LIMIT = 1`, then
set it to `None` and rerun the preview and batch cells for all eligible sessions.

F1 TV **Access/Pro/Premium entitlement** may be needed; there is no separate Pro API
key to paste here. A free F1 account alone may not grant archive access. Authenticate
locally with FastF1; never put passwords, tokens, or login cookies in notebook cells.
Opening or running the default notebook performs **no network requests**.


In [ ]:
from pathlib import Path
import sys
import json
import runpy
import subprocess
import time
from datetime import date, datetime, timedelta, timezone
from importlib.metadata import version
from zoneinfo import ZoneInfo

import fastf1
import pandas as pd
from IPython.display import display
from packaging.version import Version

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'data/archive/store.py').is_file() and (p / 'primusEnv').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook inside the F1 Primus AI project.')
expected_python = (ROOT / 'primusEnv/Scripts/python.exe').resolve()
if Path(sys.executable).resolve() != expected_python:
    raise RuntimeError(f'Select the primusEnv kernel: {expected_python}')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from data.archive.store import Archive, utc_now
from data.archive.http import PacedHTTP, ProviderBlocked, RetryDeferred
from data.archive.fastf1_extract import calendar_rows, extract_session
from data.archive.fastf1_cache import archived_event, add_original_cached_streams
from data.archive.collect import schedule_frame, worker_lock

print('Project:', ROOT)
print('Python:', sys.executable)
print('FastF1:', version('fastf1'))


## 1. Configuration

The original historical boundary remains **October 23, 2026, inclusive in Chicago**.
Change `THROUGH_DATE` if you want a different completed-session cutoff. Calendar
records may describe future sessions; those sessions are never put in the batch.
The four-hour race/two-hour other-session margin avoids assuming a just-started
session is finalized. This is conservative, not evidence of an official finish.
After load, observed timestamps are checked against the cutoff again.

No old data are seeded. New files go only to `.local/f1_2025_2026/` (gitignored).


In [ ]:
YEARS = (2025, 2026)
THROUGH_DATE = '2026-10-23'
TIMEZONE = 'America/Chicago'
SESSION_CODES = {'FP1', 'FP2', 'FP3', 'Q', 'SQ', 'S', 'R'}
ROUND_NUMBERS = None        # None = every round; example {1, 2}
SESSION_LIMIT = 1           # Pilot first. Set None for the full batch.
RUN_DOWNLOAD = True       # Explicitly enable after local setup/login.
UPGRADE_SDK = False        # Optional setup step; restart kernel after upgrading.
AUTHENTICATE_LOCALLY = True
RETRY_PARTIAL = False      # True deliberately revisits partial sessions.
MAX_ATTEMPTS = 3
STOP_AFTER_CONSECUTIVE_FAILURES = 3

OUTPUT = ROOT / '.local/f1_2025_2026'
CACHE = OUTPUT / 'fastf1-cache'
PROGRESS_FILE = OUTPUT / 'progress.json'
cutoff = datetime.combine(date.fromisoformat(THROUGH_DATE) + timedelta(days=1),
                          datetime.min.time(), ZoneInfo(TIMEZONE)).astimezone(timezone.utc)
if not YEARS or len(set(YEARS)) != len(YEARS) or any(y < 2018 or y > 2026 for y in YEARS):
    raise ValueError('Use unique years between 2018 and 2026; start with 2025 and 2026.')
if SESSION_LIMIT is not None and SESSION_LIMIT < 1:
    raise ValueError('SESSION_LIMIT must be positive or None.')
print('Years:', YEARS, '| exclusive UTC cutoff:', cutoff.isoformat())
print('Output:', OUTPUT, '| downloads enabled:', RUN_DOWNLOAD)


## 2. Authentication and SDK prerequisite

The current project runtime was FastF1 3.8.1 when this notebook was prepared.
Archive authentication requires a compatible newer SDK. In a **primusEnv** terminal,
upgrade deliberately before the first download, then restart this notebook kernel:

```
& '.\primusEnv\Scripts\python.exe' -m pip install --upgrade 'fastf1>=3.9,<4'
```

Alternatively set `UPGRADE_SDK=True` and run the next cell to perform the same
installation from this notebook. Restart the kernel afterward and set it back to
False. Do not enable download until setup and local sign-in are complete.

The requirement follows the maintainer's archive-authentication release discussion;
if that release is unavailable in your package index, stop instead of substituting
an unverified package. The existing project environment has not been upgraded by
creating this notebook. Rerun the project checks after an SDK upgrade.

Set `AUTHENTICATE_LOCALLY=True` and run the next cell to use FastF1's own login.
The login is interactive, and its token stays in FastF1's local credential storage.
The notebook does not read or display that token. Do not commit executed login output.

Sources: [FastF1 archive-access issue](https://github.com/theOehrly/Fast-F1/issues/978),
[official authentication documentation](https://github.com/theOehrly/Fast-F1/blob/main/docs/api_reference/accounts_auth.rst).

The login cell uses Jupyter input and a masked password prompt. Credentials are supplied to the official SDK only in this kernel process during sign-in, then restored/removed in a finally block. They are not written to the notebook or a .env file.


In [ ]:
def require_download_runtime():
    if Version(fastf1.__version__) != Version(version('fastf1')):
        raise RuntimeError('FastF1 changed on disk. Restart this kernel before downloading.')
    if Version(fastf1.__version__) < Version('3.9.0'):
        raise RuntimeError('Archive authentication needs FastF1 >=3.9. Upgrade in primusEnv and restart the kernel.')

def authenticate_in_notebook():
    """Prompt locally; expose credentials to the SDK only during this call."""
    import getpass
    import os
    username = os.environ.get('F1_USERNAME') or input('F1 account email: ').strip()
    password = os.environ.get('F1_PASSWORD') or getpass.getpass('F1 account password (hidden): ')
    if not username or not password:
        raise ValueError('Sign-in cancelled: email and password are required locally.')
    old_environment = {key: os.environ.get(key) for key in ('F1_USERNAME', 'F1_PASSWORD')}
    previous_args = sys.argv[:]
    try:
        os.environ['F1_USERNAME'] = username
        os.environ['F1_PASSWORD'] = password
        sys.argv = ['fastf1', 'auth', 'f1tv', '--authenticate', '--method', 'password']
        runpy.run_module('fastf1', run_name='__main__')
    finally:
        sys.argv = previous_args
        for key, value in old_environment.items():
            if value is None:
                os.environ.pop(key, None)
            else:
                os.environ[key] = value
        username = password = None

if UPGRADE_SDK:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'fastf1>=3.9,<4'], check=True)
    raise RuntimeError('SDK updated. Restart this kernel, set UPGRADE_SDK=False, then rerun setup.')

if AUTHENTICATE_LOCALLY:
    require_download_runtime()
    authenticate_in_notebook()
else:
    print('Local login has not been started. Enable AUTHENTICATE_LOCALLY when ready.')


## 3. Helpers: checkpoints, source evidence, session selection

Every real HTTP send uses the existing conservative pacer (11 seconds for FastF1
and shared SDK traffic, 21 seconds for Jolpica). SDK limiters remain enabled.
HTTP 429/Retry-After and hourly budgets are retained across notebook restarts.
A long quota delay pauses the visible batch rather than launching another worker.
We store original schedule bytes and trusted SDK parser cache bytes, not credentials.
Parquet contains original DataFrame columns plus normalized clocks/quality flags.
Parser cache frames are not original wire packets; SDK-generated fills remain flagged.


In [ ]:
def save_progress(progress):
    OUTPUT.mkdir(parents=True, exist_ok=True)
    temporary = PROGRESS_FILE.with_suffix('.tmp')
    temporary.write_text(json.dumps(progress, indent=2), encoding='utf-8')
    temporary.replace(PROGRESS_FILE)

def read_progress():
    if not PROGRESS_FILE.exists():
        return {'schema_version': 1, 'sessions': {}}
    progress = json.loads(PROGRESS_FILE.read_text(encoding='utf-8'))
    if progress.get('schema_version') != 1 or not isinstance(progress.get('sessions'), dict):
        raise ValueError('Unknown checkpoint schema; preserve it and inspect before continuing.')
    return progress

def choose_sessions(rows, end, years, codes, rounds=None):
    # Integer round ordering avoids round 10 appearing before round 2.
    selected = []
    end_ns = int(end.timestamp()) * 10**9
    for original in rows:
        row = dict(original)
        year, round_number = map(int, row['event_id'].split(':'))
        start = row.get('scheduled_start_utc_ns')
        margin = (4 if row['session_code'] == 'R' else 2) * 3600 * 10**9
        if (year in years and row['session_code'] in codes and start is not None
            and start + margin <= end_ns and (rounds is None or round_number in rounds)):
            selected.append(row)
    return sorted(selected, key=lambda row: (int(row['event_id'].split(':')[0]),
                                             int(row['event_id'].split(':')[1]),
                                             row['scheduled_start_utc_ns']))

def access_denied(http):
    return any(route['blocked_status'] in (401, 403) for route in http.status()['routes'])

def verify_published(archive, version_id):
    with archive.connect() as conn:
        row = conn.execute("SELECT status FROM session_versions WHERE version_id=?", (version_id,)).fetchone()
        if row is None or row['status'] != 'published':
            raise ValueError('Checkpoint points to an unpublished session.')
        archive._verify_files(conn, version_id)  # Same hash/row checks as atomic publication.


## 4. Download calendars and preview the exact session list

Only the two configured years are requested. Round names, session names and dates
are retained. Calendar retrieval is separate from telemetry authentication.


In [ ]:
archive = None
http = None
planned_sessions = []
if RUN_DOWNLOAD:
    require_download_runtime()
    archive = Archive(OUTPUT)
    http = PacedHTTP(OUTPUT)
    # Keep all original schedule columns in an immutable source object.
    with worker_lock(OUTPUT):
        for year in YEARS:
            url = f'https://raw.githubusercontent.com/theOehrly/f1schedule/master/schedule_{year}.json'
            response = http.get(url, provider='fastf1')
            source_id = archive.source('fastf1_schedule', url, response.content, utc_now(),
                                       url=url, endpoint_kind='calendar')
            events, sessions = calendar_rows(year, schedule_frame(response.json()))
            archive.save_calendar(events, sessions, source_id)
    with archive.connect() as conn:
        rows = conn.execute('SELECT s.*, e.event_name, e.local_race_date FROM sessions s JOIN events e USING(event_id)').fetchall()
    planned_sessions = choose_sessions(rows, cutoff, set(YEARS), SESSION_CODES, ROUND_NUMBERS)
    print(f'{len(planned_sessions)} eligible sessions, before pilot/resume filtering.')
    if planned_sessions:
        display(pd.DataFrame(planned_sessions)[['event_name', 'local_race_date', 'full_name', 'session_id']])
    else:
        print('No completed sessions match the selected years, rounds and cutoff.')
else:
    print('Preview only: no calendar or telemetry requests. Enable RUN_DOWNLOAD after setup/login.')


## 5. Download and atomically publish one session

This keeps results, entries, drivers/teams, laps/sectors/tyre compounds, status and
race-control messages, on-track weather, and each driver's separate car/position
streams **when supplied**. Known absent channels stay absent. Tyre temperature is
not supplied by normal FastF1 feeds. Circuit corner labels and high-frequency gap
streams require separate adapters and are not claimed as collected here.

Weather here is **FastF1's on-track weather**, not forecasts or Open-Meteo reanalysis.
External weather enrichment can be added after the first dataset is verified.


In [ ]:
def download_session(row, archive, http):
    session_id = row['session_id']
    event = archived_event(archive, row['event_id'])
    # Historical Sprint Shootout and Sprint Qualifying names are not interchangeable.
    session = event.get_session(row['original_provider_name'] or row['full_name'])
    CACHE.mkdir(parents=True, exist_ok=True)
    fastf1.Cache.enable_cache(str(CACHE))
    fastf1.Cache.offline_mode(False)
    with http.install_fastf1():
        load_error = None
        try:
            session.load(laps=True, telemetry=True, weather=True, messages=True)
        except SystemExit as exc:
            load_error = f'SDK exited while loading: {exc}'
        # SDK soft loaders catch network exceptions: never publish a quota-truncated session.
        if http.last_sdk_deferred is not None:
            raise http.last_sdk_deferred
        result = extract_session(session, session_id)
        cache_path = (CACHE / session.api_path.removeprefix('/static/')).resolve()
        if not cache_path.is_relative_to(CACHE.resolve()):
            raise ValueError('Unsafe SDK cache path.')
        add_original_cached_streams(session, cache_path, result, session_id)
        sources = []
        for path in sorted(cache_path.glob('*.ff1pkl')):
            if path.is_symlink() or not path.resolve().is_relative_to(CACHE.resolve()):
                raise ValueError('Unsafe parser cache file.')
            sources.append(archive.source('fastf1_parser_cache', str(path.relative_to(CACHE)),
                path.read_bytes(), utc_now(), endpoint_kind=path.stem,
                availability_basis='hindsight_download_original_publication_unverified'))
        # Preserve fetched evidence even if it is ineligible for this cutoff.
        observed_end = result['metadata'].get('observed_end_utc_ns')
        if observed_end is not None and observed_end >= int(cutoff.timestamp()) * 10**9:
            raise ValueError('Session observations cross the configured cutoff; not published.')
        if access_denied(http):
            raise ProviderBlocked('fastf1', 'timing_archive', 401)
        if not result['tables'].get('laps'):
            raise ValueError(load_error or 'No lap data returned; session is not complete.')
        result['metadata']['collection_scope'] = {'years': list(YEARS), 'cutoff_utc': cutoff.isoformat()}
        result['metadata']['load_error'] = load_error
        version_id = archive.publish_session(session_id, result['tables'], result['datasets'], sources, result['metadata'])
        coverage = result['metadata']['completeness']
        required = ('laps', 'car', 'positions', 'weather', 'race_control')
        status = 'complete' if not load_error and all(coverage[k]['status'] == 'available' for k in required) else 'partial'
        return {'status': status, 'version_id': version_id, 'coverage': coverage,
                'laps': len(result['tables']['laps']), 'saved_at_utc': utc_now()}


## 6. Visible batch loop: pilot, resume and bounded failures

Interrupt the kernel to stop. A finished session is checkpointed immediately;
an interrupted session safely resumes from the SDK cache. Completed checkpoints
are reused only after verifying their published files. A second notebook cannot
write the same archive concurrently. Nothing runs after the cell stops.

An authentication denial stops the batch immediately. After repairing local login,
explicitly run `http.reset_denied('fastf1')` before retrying the batch cell; quota
history is retained. A quota delay prints its next retry time and stops this run.
For a full collection, set `SESSION_LIMIT=None`, rerun configuration and calendar,
then rerun this cell. Use `RETRY_PARTIAL=True` to retry incomplete saved sessions.


In [ ]:
if RUN_DOWNLOAD:
    if archive is None or http is None:
        raise RuntimeError('Run the calendar cell first.')
    progress = read_progress()
    consecutive_failures = 0
    attempted = 0
    with worker_lock(OUTPUT):
        for index, row in enumerate(planned_sessions, 1):
            session_id = row['session_id']
            previous = progress['sessions'].get(session_id, {})
            if previous.get('status') == 'complete' or (previous.get('status') == 'partial' and not RETRY_PARTIAL):
                verify_published(archive, previous['version_id'])
                continue
            if SESSION_LIMIT is not None and attempted >= SESSION_LIMIT:
                break
            attempted += 1
            print(f"[{index}/{len(planned_sessions)}] {row['event_name']} — {row['full_name']}", flush=True)
            halt = False
            for attempt in range(1, MAX_ATTEMPTS + 1):
                try:
                    record = download_session(row, archive, http)
                    progress['sessions'][session_id] = record
                    consecutive_failures = 0
                    print(f"  {record['status']}: {record['laps']} laps; version {record['version_id'][:12]}", flush=True)
                    break
                except ProviderBlocked:
                    progress['sessions'][session_id] = {'status': 'blocked', 'reason': 'Authentication/access denied; local login or entitlement required.', 'saved_at_utc': utc_now()}
                    print('Access denied. Batch stopped; no further sessions will be requested.')
                    halt = True
                    break
                except RetryDeferred as exc:
                    retry_utc = datetime.fromtimestamp(exc.retry_at, timezone.utc).isoformat()
                    progress['sessions'][session_id] = {'status': 'deferred', 'retry_after_utc': retry_utc, 'reason': exc.reason}
                    print('Provider requested a pause until', retry_utc, '— rerun this cell after that time.')
                    halt = True
                    break
                except Exception as exc:
                    if attempt == MAX_ATTEMPTS:
                        consecutive_failures += 1
                        progress['sessions'][session_id] = {'status': 'failed', 'error_type': type(exc).__name__, 'reason': 'Session failed after bounded retries; inspect SDK warnings above.', 'saved_at_utc': utc_now()}
                        print(f'  Failed after {attempt} attempts ({type(exc).__name__}).')
                    else:
                        print(f'  Retry {attempt}/{MAX_ATTEMPTS} in {30 * attempt} seconds ({type(exc).__name__}).')
                        time.sleep(30 * attempt)
            save_progress(progress)
            if halt or consecutive_failures >= STOP_AFTER_CONSECUTIVE_FAILURES:
                break
    print('Batch cell finished. Checkpoints:', PROGRESS_FILE)
    if progress['sessions']:
        display(pd.DataFrame.from_dict(progress['sessions'], orient='index')[['status']])
else:
    print('Download loop disabled. No worker has been started.')


## 7. Verify the database, actual coverage, and Parquet samples

The physical tables retain revisions. `v_current_laps` selects one whole rich lap
version per session; counts below are not summed across duplicate revisions.
Data quality is a separate gate from successful HTTP requests. Historical downloads
remain hindsight observations, not proof that those values were available before a
past race. Keep that distinction when training or evaluating predictions.


In [ ]:
if (OUTPUT / 'f1.sqlite3').exists():
    checked_archive = Archive(OUTPUT)
    with checked_archive.connect() as conn:
        assert conn.execute('PRAGMA integrity_check').fetchone()[0] == 'ok'
        assert not conn.execute('PRAGMA foreign_key_check').fetchall()
        versions = [r[0] for r in conn.execute("SELECT version_id FROM session_versions WHERE status='published'")]
        counts = {table: conn.execute(f'SELECT count(*) FROM {table}').fetchone()[0]
                  for table in ('events', 'sessions', 'drivers', 'teams', 'v_current_laps', 'weather_observations', 'dataset_files')}
        coverage_rows = pd.read_sql_query('SELECT v.session_id,c.dataset_kind,c.status,c.rows,c.reason FROM coverage c JOIN session_versions v USING(version_id)', conn)
        sample = conn.execute('SELECT relative_path FROM dataset_files ORDER BY published_at_utc DESC LIMIT 1').fetchone()
    for version_id in versions:
        verify_published(checked_archive, version_id)
    print('SQLite/FKs and published source/Parquet hashes verified.')
    display(pd.DataFrame([counts]))
    display(coverage_rows)
    if sample:
        display(pd.read_parquet(OUTPUT / sample['relative_path']).head())
else:
    print('No new dataset has been loaded yet. Complete the pilot before starting Streamlit phase 1.')


## 8. Proceed to Streamlit one phase at a time

Review `docs/streamlit-phased-restart.md`. Phase 0 is this collection/validation
gate. Phase 1 connects the verified database and renders Race Overview. Replay,
comparisons, strategy, predictions, validation and Primus AI follow separately.
No UI implementation or bulk download is implied by opening this notebook.

The old 2018–2026 archive, caches and session/snapshot exports were deleted at your
request. This notebook does not restore or silently reuse them. Existing experiment
history is separate and is not evidence that the new dataset has been loaded.
